In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
df = pd.read_csv("merged_all_datasets.csv")

In [3]:
df["tags"] = df["tags"].apply(lambda x: x.split())

df["num_tags"] = df["tags"].apply(len)

# فصل البيانات
df_1 = df[df["num_tags"] == 1].copy()
df_2 = df[df["num_tags"] == 2].copy()
df_3_plus = df[df["num_tags"] >= 3].copy()

df_1_sample = df_1.sample(
    n=min(1000000, len(df_1)),
    random_state=42
).copy()

df_2_sample = df_2.sample(
    n=min(350000, len(df_2)),
    random_state=42
).copy()

df_final = pd.concat([df_1_sample, df_2_sample, df_3_plus], ignore_index=True)

df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)



X = df_final["text"]
y = df_final["tags"]

In [4]:
# 5) تقسيم البيانات
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

In [5]:
# 6) MultiLabelBinarizer
mlb = MultiLabelBinarizer()

y_train_bin = mlb.fit_transform(y_train)
y_val_bin = mlb.transform(y_val)
y_test_bin = mlb.transform(y_test)



In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier

# 1) TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=100,
    max_df=0.9,
    max_features=150000,
    sublinear_tf=True
)

#  fit فقط على train
X_train_vec = vectorizer.fit_transform(X_train)

# transform للباقي
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print("شكل X_train:", X_train_vec.shape)

شكل X_train: (1196362, 17868)


In [7]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score



model = OneVsRestClassifier(
    LogisticRegression(
        solver="saga",
        max_iter=500,
        random_state=42,
        n_jobs=-1
    ),
    n_jobs=-1
)

model.fit(X_train_vec, y_train_bin)




,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...solver='saga')
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",-1
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=No

In [8]:
val_scores = model.predict_proba(X_val_vec)

if isinstance(val_scores, list):
    val_scores = np.array([p[:, 1] for p in val_scores]).T


In [9]:
def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

In [10]:
def evaluate_at_k(y_true, y_scores, k):
    y_pred_k = top_k_binary_predictions(y_scores, k)

    precision = precision_score(y_true, y_pred_k, average="micro", zero_division=0)
    recall = recall_score(y_true, y_pred_k, average="micro", zero_division=0)
    f1 = f1_score(y_true, y_pred_k, average="micro", zero_division=0)

    return precision, recall, f1

In [12]:
import numpy as np
for k in [1, 2, 3,4,5]:
    p, r, f = evaluate_at_k(y_val_bin, val_scores, k)

    print(f"\n===== Top-{k} Metrics =====")
    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}:    {r:.4f}")
    print(f"F1@{k}:        {f:.4f}")



===== Top-1 Metrics =====
Precision@1: 0.8617
Recall@1:    0.5992
F1@1:        0.7069

===== Top-2 Metrics =====
Precision@2: 0.5883
Recall@2:    0.8182
F1@2:        0.6845

===== Top-3 Metrics =====
Precision@3: 0.4315
Recall@3:    0.9001
F1@3:        0.5833

===== Top-4 Metrics =====
Precision@4: 0.3365
Recall@4:    0.9360
F1@4:        0.4950

===== Top-5 Metrics =====
Precision@5: 0.2746
Recall@5:    0.9549
F1@5:        0.4266


In [13]:

import numpy as np

test_scores = model.predict_proba(X_test_vec)

# إذا رجعت list، نحولها إلى array
if isinstance(test_scores, list):
    test_scores = np.array([p[:, 1] for p in test_scores]).T

In [14]:
for k in [1, 2, 3, 4]:
    p, r, f = evaluate_at_k(y_test_bin, test_scores, k)

    print(f"\n===== Top-{k} Test Metrics =====")
    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}:    {r:.4f}")
    print(f"F1@{k}:        {f:.4f}")


===== Top-1 Test Metrics =====
Precision@1: 0.8621
Recall@1:    0.5992
F1@1:        0.7070

===== Top-2 Test Metrics =====
Precision@2: 0.5889
Recall@2:    0.8186
F1@2:        0.6850

===== Top-3 Test Metrics =====
Precision@3: 0.4320
Recall@3:    0.9008
F1@3:        0.5840

===== Top-4 Test Metrics =====
Precision@4: 0.3367
Recall@4:    0.9362
F1@4:        0.4953


In [15]:
df["tags"] = df["tags"].apply(lambda x: x.split())

df["num_tags"] = df["tags"].apply(len)

# فصل البيانات
df_1 = df[df["num_tags"] == 1].copy()
df_2 = df[df["num_tags"] == 2].copy()
df_3_plus = df[df["num_tags"] >= 3].copy()

df_1_sample = df_1.sample(
    n=min(1000000, len(df_1)),
    random_state=42
).copy()

df_2_sample = df_2.sample(
    n=min(320000, len(df_2)),
    random_state=42
).copy()

df_final = pd.concat([df_1_sample, df_2_sample, df_3_plus], ignore_index=True)

df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

X = df_final["text"]
y = df_final["tags"]

AttributeError: 'list' object has no attribute 'split'

In [ ]:
# 5) تقسيم البيانات
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

In [ ]:
# 6) MultiLabelBinarizer
mlb = MultiLabelBinarizer()

y_train_bin = mlb.fit_transform(y_train)
y_val_bin = mlb.transform(y_val)
y_test_bin = mlb.transform(y_test)



In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier

# 1) TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=100,
    max_df=0.9,
    max_features=100000,
    sublinear_tf=True
)

# ⚠️ fit فقط على train
X_train_vec = vectorizer.fit_transform(X_train)

# transform للباقي
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print("شكل X_train:", X_train_vec.shape)

شكل X_train: (1196362, 17868)


In [17]:
import joblib
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, jaccard_score

model = OneVsRestClassifier(
    LinearSVC(
        C=1,
        max_iter=13000,
        random_state=42
    ),
    n_jobs=-1
)

model.fit(X_train_vec, y_train_bin)



,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LinearSVC(C=1...ndom_state=42)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",-1
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the 

In [18]:
import numpy as np
val_scores = model.decision_function(X_val_vec)
val_scores = np.asarray(val_scores)
print("val_scores shape:", val_scores.shape)


val_scores shape: (149545, 63)


In [19]:
def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

In [20]:
def evaluate_at_k(y_true, y_scores, k):
    y_pred_k = top_k_binary_predictions(y_scores, k)

    precision = precision_score(y_true, y_pred_k, average="micro", zero_division=0)
    recall = recall_score(y_true, y_pred_k, average="micro", zero_division=0)
    f1 = f1_score(y_true, y_pred_k, average="micro", zero_division=0)

    return precision, recall, f1

In [21]:
from sklearn.metrics import precision_score, recall_score, f1_score

for k in [1, 2, 3,4]:
    p, r, f = evaluate_at_k(y_val_bin, val_scores, k)

    print(f"\n===== Top-{k} Metrics =====")
    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}:    {r:.4f}")
    print(f"F1@{k}:        {f:.4f}")



===== Top-1 Metrics =====
Precision@1: 0.8635
Recall@1:    0.6005
F1@1:        0.7084

===== Top-2 Metrics =====
Precision@2: 0.5881
Recall@2:    0.8179
F1@2:        0.6842

===== Top-3 Metrics =====
Precision@3: 0.4303
Recall@3:    0.8976
F1@3:        0.5817

===== Top-4 Metrics =====
Precision@4: 0.3348
Recall@4:    0.9313
F1@4:        0.4926


In [22]:
from sklearn.metrics import precision_score, recall_score, f1_score

# 1) استخراج scores من LinearSVC على test
test_scores = model.decision_function(X_test_vec)
test_scores = np.asarray(test_scores)

In [23]:
for k in [1, 2, 3, 4]:
    p, r, f = evaluate_at_k(y_test_bin, test_scores, k)

    print(f"\n===== Top-{k} Test Metrics =====")
    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}:    {r:.4f}")
    print(f"F1@{k}:        {f:.4f}")


===== Top-1 Test Metrics =====
Precision@1: 0.8641
Recall@1:    0.6006
F1@1:        0.7087

===== Top-2 Test Metrics =====
Precision@2: 0.5882
Recall@2:    0.8176
F1@2:        0.6842

===== Top-3 Test Metrics =====
Precision@3: 0.4303
Recall@3:    0.8972
F1@3:        0.5816

===== Top-4 Test Metrics =====
Precision@4: 0.3347
Recall@4:    0.9306
F1@4:        0.4924
